# Lab: Regression Tree

*In this lab, we will build a **Regression Tree** model using scikit-learn, guided by the [Foundational Methodology for Data Science](../../../05_methodology/). While real-world projects require comprehensive documentation at each stage, this lab focuses on a streamlined, practical approach to demonstrate the core concepts and workflow. Therefore we will settle with the summaries of hypothetical stage reports.*

---

> ### 📝 1. Business Understanding Report (Summary)
>
> - **Business Context:** This project addresses a real-estate analytics scenario for a hypothetical company, **"Housewise Analytics"**, operating in the California housing market. California's housing market is characterized by diverse neighborhoods, rapidly changing prices, and growing demand for data-driven decision-making among buyers, sellers, and realtors.
> - **Business Problem:** Stakeholders currently rely on broad averages or intuition, lacking a precise and explainable method to estimate home prices for specific neighborhoods and property characteristics. This uncertainty leads to lost sales opportunities, mispricing, and inefficient market decisions.
> - **Project Goal:** To build a **predictive model** that uses available district-level features (such as location, median income, number of rooms, and more) to accurately estimate median home values in California. The objective is to empower more confident and data-driven pricing decisions that benefit both buyers and sellers.

---

> ### 📝 2. Analytic Approach Report (Summary)
>
> - **Problem Type:** *Supervised Regression* - the goal is to predict a **continuous numerical value** (median house value) from district-level features.
> - **Model Selection:** For this lab, the candidate model is a **Regression Tree (Decision Tree Regressor)**. This approach was chosen because it can naturally handle non-linear relationships and feature interactions, which are common in housing data. (Note: In previous labs, simple and multiple linear regression were already explored.)
> - **Evaluation Metrics:**
>   - **R-squared (R²) and Adjusted R-squared:** To measure how much of the variance in house values the model can explain, while adjusting for the number of features.
>   - **RMSE (Root Mean Squared Error):** To provide a direct and interpretable measure of average prediction errors, using the original units of the target (house value in $100,000s).
>   - **Feature Importance:** To identify which features (e.g., median income, rooms, latitude/longitude) have the largest impact on price predictions.
> - **Motivation for Model Selection:** In a real-world analytic workflow, **Exploratory Data Analysis (EDA)** would precede model building. EDA (including scatterplots and residual analysis) often uncovers non-linear relationships between predictors and housing prices, motivating the use of regression trees. Regression trees can capture threshold, interaction, and piecewise patterns that linear models miss.
>
> ⚠️ **Caution:** For clarity and focus, this lab considers only a regression tree. In real-world projects, it is best practice to compare several modeling approaches (including advanced ensembles and regularized models) to select the most robust solution.

---

> ### 📝 3. Data Requirements Report (Summary)
>
> - **Data Source:** The lab will use the **California Housing dataset** (available via scikit-learn), which includes data aggregated from 20,000+ block groups across California.
> - **Features (Independent Variables):**
>   - Median income (`med_income`)
>   - Average number of rooms (`avg_rooms`)
>   - Average number of bedrooms (`avg_bedrooms`)
>   - Population (`population`)
>   - Average house occupancy (`avg_occup`)
>   - Latitude (`lat`)
>   - Longitude (`long`)
> - **Target (Dependent Variable):**
>   - Median house value (`med_value`) expressed in $100,000s.
> - **Granularity & Privacy:** Each row represents aggregated statictics for a California census block group (district), not individual households. The dataset is fully anonymized and contains no sensitive or personally identifiable information (PII).

---

## Stage 4: Data Collection

In [61]:
# Necessary imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.datasets import fetch_california_housing

# Configurations
plt.style.use("fivethirtyeight")
sns.set_theme(style="white", palette="colorblind")

### 4.1 Extraction

In [62]:
# Extract data from the source system
try:
    raw_data = fetch_california_housing()
    print(f"Data has been extracted from the source system as '{type(raw_data)}'")
except Exception as e:
    print(f"An error occurred while extracting data: {e}")

Data has been extracted from the source system as '<class 'sklearn.utils._bunch.Bunch'>'


In [63]:
# Create a DataFrame with the original, ontouched column names
raw_df = pd.DataFrame(data=raw_data.data, columns=raw_data.feature_names)
raw_df['target'] = raw_data.target

# Define and create the raw data directory
raw_data_dir = Path("../data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# Define the file path and save the raw DataFrame
raw_data_path = raw_data_dir / "california_housing_raw_v1.csv"
raw_df.to_csv(raw_data_path, index=False)

print(f"Extracted data has {raw_df.shape[0]} samples and {raw_df.shape[1]} features, with {raw_df.isnull().values.sum()} missing values out of {raw_df.size}.")
print(f"Raw, untouched data has been saved to: {raw_data_path}\n")
raw_df.head()

Extracted data has 20640 samples and 9 features, with 0 missing values out of 185760.
Raw, untouched data has been saved to: ../data/raw/california_housing_raw_v1.csv



,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422


### 4.2. Tranformation

In [64]:
# Read the raw data
interim_df = pd.read_csv(raw_data_path)

In [65]:
# Rename columns
interim_df = interim_df.rename(columns={
    'MedInc':'med_income',
    'HouseAge': 'age',
    'AveRooms': 'avg_rooms',
    'AveBedrms': 'avg_bedrooms',
    'Population': 'population',
    'AveOccup': 'avg_occup',
    'Latitude': 'lat',
    'Longitude': 'long',
    'target': 'med_value'
})

In [66]:
# Manually check the minimum and maximum values for all numerical columns for any abnormalities
print("\n--- Min/Max Summary of Numerical Data ---")
display(interim_df.describe().loc[['min', 'max']]) 


--- Min/Max Summary of Numerical Data ---


,med_income,age,avg_rooms,avg_bedrooms,population,avg_occup,lat,long,med_value
min,0.4999,1.0,0.846154,0.333333,3.0,0.692308,32.54,-124.35,0.14999
max,15.0001,52.0,141.909091,34.066667,35682.0,1243.333333,41.95,-114.31,5.00001


* 📌 There seems to be unusually high values in average rooms, average bedrooms, population, and average occupancy. However, at this stage we can not be 100% sure that these are data entry errors, as they may represent legitimate cases such as hotels, group living environments, or unique properties.

### 4.3. Loading

In [67]:
# Define and create the interim data directory
interim_data_dir = Path("../data/interim")
interim_data_dir.mkdir(parents=True, exist_ok=True)

# Define the interim file path and save the transformed DataFrame
interim_data_path = interim_data_dir / "california_housing_interim_v1.parquet"
interim_df.to_parquet(interim_data_path, index=False)

### 4.4. Verification

In [68]:
# Load the final interim data and verify its contents
try:
    cal_housing_df = pd.read_parquet(interim_data_path)
    print(f"Verification successful. The following DataFrame is ready for analysis with {cal_housing_df.shape[0]} samples and {cal_housing_df.shape[1]} features:")
    display(cal_housing_df.sample(5))
except Exception as e:
    print(f"An error occurred while verifying the interim data: {e}")

Verification successful. The following DataFrame is ready for analysis with 20640 samples and 9 features:


,med_income,age,avg_rooms,avg_bedrooms,population,avg_occup,lat,long,med_value
14881,4.1250,37.0,5.525000,0.975000,612.0,2.550000,32.63,-117.06,1.602
18408,5.0772,17.0,5.243968,0.987936,2693.0,3.609920,37.27,-121.80,2.215
8726,4.7308,35.0,5.666667,0.971831,601.0,2.821596,33.83,-118.37,3.534
8138,4.8750,36.0,5.609562,0.996016,702.0,2.796813,33.83,-118.10,2.225
15824,2.7173,52.0,4.245161,1.129032,935.0,2.010753,37.75,-122.42,3.000



---

> ### 📝 4. Data Collection Report (Summary)
> 
> - **Extraction:** The California Housing dataset was retrieved from scikit-learn's repository and saved in its original, ontouched form as a raw `.csv` file for reproducibility and provenance.
> - **Transformation:**
>   - All columns are renamed as decided in the Data Reqirements Report (Summary).
>   - Data quality checks revealed unusually high values in average rooms (`avg_rooms`), average bedrooms (`avg_bedrooms`), population (`population`), and average occupancy (`avg_occup`). Those potential outliers were **not** automatically removed during transformation, as they may represent legitimate cases such as hotels, group living environments, or unique properties. Determining their treatment is left for the data scientist at the EDA stage, based on domain knowledge and analytic objectives.
> - **Loading:** The standardized dataset was saved as a `.parquet` file in the `/data/interim` directory, providing analysis-ready starting point for all subsequent work.
> - **Verification:** The interim DataFrame contains 20,640 rows and 9 columns, with all variable names in snake_case and no missing values detected.

---

## Stage 5: Data Understanding

### 5.1. Preparation for Exploratory Data Analyis (EDA)